In [1]:
import sys

sys.path.append("../src")

In [2]:
import os

import optuna
import pandas as pd
from sklearn.metrics import classification_report, roc_auc_score
from tqdm.notebook import tqdm

from experiments.ensemble_evaluation import EnsembleEvaluation
from utils import *

In [3]:
paths = Paths("../configs/paths.json")

## Find the best trial per outer fold

In [4]:
def get_best_trial_number(fold_dir):
    path = os.path.join(fold_dir, "optuna.db")
    study = optuna.load_study(study_name=None, storage=f"sqlite:///{path}")
    return study.best_trial.number

## Check for any missing folds

In [7]:
folds_to_check = []
for fold in tqdm(
    sorted(Path("../results/optimization/").glob("fold_*/")),
    leave=False,
    desc="Outer Folds",
):
    # trial_number = get_best_trial_number(fold)
    for trial_number in fold.glob("trial_*/"):
        trial_number = trial_number.stem.split("_")[1]
        root = fold / f"trial_{trial_number}" / "models"
        for i in range(6):
            prefix = f"TransformerEngagement_{i}"
            if os.path.exists(root / f"{prefix}.pth") and os.path.exists(root / f"{prefix}_engagement_val.csv"):
                continue
            folds_to_check.append([fold.stem, f'trial_{trial_number}', i])

In [8]:
folds_to_check

## Evaluate each outer fold

In [131]:
all_folds_data = {"soft":[], "hard":[], "weighted":[]}
for fold in tqdm(
    sorted(Path("../results/optimization/BCE").glob("fold_*/")),
    leave=False,
    desc="Outer Folds",
):
    best_trial_number = get_best_trial_number(fold)

    eval_obj = EnsembleEvaluation(
        data_root=paths["FeaturesRoot"],
        experiment_root=fold / f"trial_{best_trial_number}",
        splits_root=paths["SplitsRoot"] / fold.stem,
    )

    df = eval_obj.generate_predictions_report().dropna()
    y_true = df.y_true_binary.astype(int)
    for mode in ["soft", "hard", "weighted"]:
        y_prob = df[mode]
        y_pred = (y_prob.apply(lambda x: x > 0.5)).astype(int)
        report = pd.DataFrame(
            classification_report(y_true, y_pred, output_dict=True)
        ).assign(roc_auc=roc_auc_score(y_true, y_prob))
        all_folds_data[mode].append(report.assign(fold=fold.stem))

Outer Folds:   0%|          | 0/6 [00:00<?, ?it/s]

[I 2024-12-23 15:50:56,153] Study name was omitted but trying to load 'webcam_engagement_classifier' because that was the only study found in the storage.

Fold:   0%|                                               | 0/6 [00:00<?, ?it/s]/home/ammar/miniconda3/envs/torch/lib/python3.12/site-packages/torch/nn/modules/transformer.py:286: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")

Fold:  17%|██████▌                                | 1/6 [00:07<00:38,  7.68s/it]/home/ammar/miniconda3/envs/torch/lib/python3.12/site-packages/torch/nn/modules/transformer.py:286: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_spa

In [132]:
# FOCAL
from IPython.display import display
for mode in ["soft", "hard", "weighted"]:
    df = pd.concat(all_folds_data[mode]).reset_index().rename(columns={"index":"metric"}).drop(["fold"], axis="columns")
    mean_df = df.groupby("metric").mean()
    std_df = df.groupby("metric").std()
    print(mode)
    display(combine_avg_std_tables_latex(mean_df, std_df))
    print()

soft
Accuracy: 0.8392425374040829 ± 0.11340736841651329
ROC-AUC: 0.5031166654131426 ± 0.16525989580005282


,Low,High,Avg.
metric,,,
precision,0.09 ± 0.15,0.87 ± 0.1,0.48 ± 0.11
recall,0.08 ± 0.14,0.96 ± 0.05,0.52 ± 0.06
f1-score,0.08 ± 0.14,0.91 ± 0.07,0.5 ± 0.09
support,13.5 ± 7.18,90.83 ± 15.85,104.33 ± 10.23



hard
Accuracy: 0.7437614650747125 ± 0.15961197499522173
ROC-AUC: 0.5086361586361586 ± 0.09522095951366588


,Low,High,Avg.
metric,,,
precision,0.07 ± 0.1,0.87 ± 0.1,0.47 ± 0.05
recall,0.18 ± 0.33,0.84 ± 0.2,0.51 ± 0.1
f1-score,0.08 ± 0.12,0.84 ± 0.12,0.46 ± 0.03
support,13.5 ± 7.18,90.83 ± 15.85,104.33 ± 10.23



weighted
Accuracy: 0.8294557542903732 ± 0.10908569232345751
ROC-AUC: 0.4973779307754036 ± 0.17744267814496834


,Low,High,Avg.
metric,,,
precision,0.07 ± 0.12,0.86 ± 0.1,0.47 ± 0.09
recall,0.07 ± 0.11,0.95 ± 0.06,0.51 ± 0.05
f1-score,0.07 ± 0.11,0.9 ± 0.07,0.48 ± 0.07
support,13.5 ± 7.18,90.83 ± 15.85,104.33 ± 10.23


In [44]:
# BCE
from IPython.display import display
for mode in ["soft", "hard", "weighted"]:
    df = pd.concat(all_folds_data[mode]).reset_index().rename(columns={"index":"metric"}).drop(["fold"], axis="columns")
    mean_df = df.groupby("metric").mean()
    std_df = df.groupby("metric").std()
    print(mode)
    display(combine_avg_std_tables_latex(mean_df, std_df))
    print()

soft
Accuracy: 0.6795836827877705 ± 0.13759662743053078
ROC-AUC: 0.65011738439288 ± 0.13582491439156177


,Low,High,Avg.
metric,,,
precision,0.23 ± 0.15,0.91 ± 0.04,0.57 ± 0.07
recall,0.54 ± 0.23,0.69 ± 0.17,0.62 ± 0.12
f1-score,0.31 ± 0.17,0.77 ± 0.13,0.54 ± 0.11
support,13.86 ± 6.62,88.57 ± 15.66,102.43 ± 10.61



hard
Accuracy: 0.6650180390398309 ± 0.09480072624830825
ROC-AUC: 0.6003886974677984 ± 0.13237490206914515


,Low,High,Avg.
metric,,,
precision,0.22 ± 0.16,0.91 ± 0.04,0.56 ± 0.09
recall,0.53 ± 0.24,0.67 ± 0.1,0.6 ± 0.13
f1-score,0.3 ± 0.2,0.77 ± 0.08,0.53 ± 0.11
support,13.86 ± 6.62,88.57 ± 15.66,102.43 ± 10.61



weighted
Accuracy: 0.6765763477488622 ± 0.1433147096983032
ROC-AUC: 0.6492471522994929 ± 0.1383262615217535


,Low,High,Avg.
metric,,,
precision,0.23 ± 0.16,0.9 ± 0.05,0.57 ± 0.08
recall,0.53 ± 0.25,0.69 ± 0.18,0.61 ± 0.13
f1-score,0.3 ± 0.18,0.77 ± 0.14,0.54 ± 0.12
support,13.86 ± 6.62,88.57 ± 15.66,102.43 ± 10.61


In [48]:
# BCE
for i in range(7):
    print(i, all_folds_data["soft"][i]["macro avg"]["f1-score"])

0 0.6828399122807017
1 0.4344390142456067
2 0.6353378285399752
3 0.4441176470588235
4 0.5337513061650992
5 0.644515328725855
6 0.4352035749751738


In [159]:
tn = get_best_trial_number("../results/optimization/FOCAL/fold_3/")
show_results(f"../results/optimization/FOCAL/fold_3/trial_{tn}/models", suffix="val")

[I 2024-12-23 17:20:20,164] Study name was omitted but trying to load 'webcam_engagement_classifier' because that was the only study found in the storage.


Accuracy: 0.8670394732192485 ± 0.06977120711581301


,Low,High,Avg.
metric,,,
precision,0.48 ± 0.41,0.89 ± 0.07,0.68 ± 0.18
recall,0.22 ± 0.24,0.97 ± 0.05,0.59 ± 0.1
f1-score,0.25 ± 0.25,0.92 ± 0.04,0.59 ± 0.12
support,14.17 ± 7.52,87.83 ± 21.35,102.0 ± 22.87


In [161]:
fold = Path("../results/optimization/FOCAL/fold_3/")
best_trial_number = get_best_trial_number(fold)

eval_obj = EnsembleEvaluation(
    data_root=paths["FeaturesRoot"],
    experiment_root=fold / f"trial_{best_trial_number}",
    splits_root=paths["SplitsRoot"] / fold.stem,
)

df = eval_obj.generate_predictions_report().dropna()
y_true = df.y_true_binary.astype(int)
results = []
for mode in ["soft", "hard", "weighted"]:
    y_prob = df[mode]
    y_pred = (y_prob.apply(lambda x: x > 0.5)).astype(int)
    report = pd.DataFrame(
        classification_report(y_true, y_pred, output_dict=True)
    ).assign(roc_auc=roc_auc_score(y_true, y_prob))
    results.append(report.assign(fold=fold.stem))

[I 2024-12-23 17:21:24,528] Study name was omitted but trying to load 'webcam_engagement_classifier' because that was the only study found in the storage.


In [162]:
results[0]

,0,1,accuracy,macro avg,weighted avg,roc_auc,fold
precision,0.0,0.876289,0.809524,0.438144,0.776141,0.355735,fold_3
recall,0.0,0.913978,0.809524,0.456989,0.809524,0.355735,fold_3
f1-score,0.0,0.894737,0.809524,0.447368,0.792481,0.355735,fold_3
support,12.0,93.000000,0.809524,105.000000,105.000000,0.355735,fold_3
